# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [ ]:
# This cell sets up a conda environment with Python 3.11, which is required for vLLM.
#
# IMPORTANT:
# 1. You must have Conda (or Miniconda) installed on your server.
# 2. Run this cell ONCE. After it completes, RESTART the Jupyter kernel.
# 3. After restarting, select the new kernel: "Python 3.11 (cse151b)"
#
import os
import shutil

# --- Check for Conda ---
if not shutil.which("conda"):
    print("❌ Conda is not installed or not in your PATH.")
    print("Please install Miniconda first. In your server's terminal, run:")
    print("wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh")
    print("bash Miniconda3-latest-Linux-x86_64.sh")
    print("\nAfter installation, close and reopen your terminal, then restart the Jupyter server.")
    raise RuntimeError("Conda not found. Please install it and restart.")
else:
    print("✅ Conda found. Proceeding with environment setup...")

ENV_NAME = "cse151b_py311"

# Create the conda environment with Python 3.11
# It's good practice to remove the old one if it exists for a clean install
print(f"--- Removing old environment '{ENV_NAME}' if it exists ---")
!conda env remove -n {ENV_NAME} -y
print(f"--- Creating new environment '{ENV_NAME}' with Python 3.11 and pip ---")
!conda create -n {ENV_NAME} python=3.11 pip -y

# Install PyTorch with the correct CUDA version (12.1) via conda
# This is the most reliable way to get GPU support working.
print("--- Installing PyTorch for CUDA 12.1 ---\n")
!conda install -n {ENV_NAME} pytorch torchvision torchaudio pytorch-cuda=12.1 -c pytorch -c nvidia -y

# Install Jupyter and the ipykernel with conda for better environment integration
print("\n--- Installing Jupyter and ipykernel ---\n")
!conda install -n {ENV_NAME} jupyter ipykernel -y

# Install other packages using pip within the new environment. We use `conda run` to ensure
# that pip is executed within the correct conda environment.
# Using `python -m pip` is a more robust way to invoke pip.
print("\n--- Installing base packages with pip ---\n")
!conda run -n {ENV_NAME} python -m pip install \
    transformers tqdm bitsandbytes accelerate \
    sympy numpy 'antlr4-python3-runtime==4.11.1'

# vLLM is installed separately as it can be sensitive.
# Since your server has a very new GPU (NVIDIA RTX 6000 Blackwell) and CUDA 13,
# the standard pre-built wheels for vLLM (which target CUDA 12.1) will not work.
#
# The solution is to build vLLM from source. This command will download the
# latest version of vLLM and compile it directly on your server, ensuring it is
# compatible with your specific hardware and drivers.
#
# WARNING: This process can take a very long time (30-60 minutes or more).
# Please be patient and let it run to completion.
print("\n--- Installing vLLM from source (this will take a long time!) ---\n")
!conda run -n {ENV_NAME} python -m pip install git+https://github.com/vllm-project/vllm.git

# Install the new environment as a Jupyter kernel
print("\n--- Installing Jupyter kernel ---\n")
KERNEL_DISPLAY_NAME = "Python 3.11 (cse151b)"
!conda run -n {ENV_NAME} python -m ipykernel install --user --name {ENV_NAME} --display-name "{KERNEL_DISPLAY_NAME}"

print("\n✅✅✅ SETUP COMPLETE ✅✅✅")
print("--> 1. RESTART THE KERNEL using the 'Kernel > Restart Kernel' menu option.")
print("--> 2. REFRESH your browser window to ensure the new kernel appears.")
print(f"--> 3. CHANGE the kernel in the top-right to: '{KERNEL_DISPLAY_NAME}'")
print("--> 4. Run the 'Verify Environment' cell below to confirm everything is working.")

### Run the cell below every time to activate the installed environment. 

In [ ]:
# NOTE: This cell is not necessary because we are not using `source activate`.
# The previous cell installed a Jupyter kernel named "Python 3.11 (cse151b)".
# After restarting, you should switch to that kernel from the menu above.
# This automatically "activates" the environment for the entire notebook.

# !source ./.venv/bin/activate

## 1a. Verify Environment

After restarting and selecting the **`Python 3.11 (cse151b)`** kernel, run the cell below. 

You should see:
1. A Python path that includes `cse151b_py311`.
2. `transformers`, `torch`, and `vllm` listed in the packages.

If you don't, you have not switched to the correct kernel. Please do so before proceeding.

In [ ]:
import sys
import os

print(f"🐍 Python Executable: {sys.executable}")

# Check if the correct kernel is active
if 'cse151b_py311' in os.path.normpath(sys.executable):
    print("✅ Correct kernel is active.")
else:
    print("\n❌ WRONG KERNEL. The notebook is NOT running in the 'cse151b_py311' environment.")
    print("   Please restart the kernel, then select 'Python 3.11 (cse151b)' from the kernel menu in the top-right.")
    # This error will stop execution
    raise RuntimeError("Incorrect Jupyter kernel selected. Please follow the instructions above.")

print("\nChecking for key packages...")
try:
    import transformers
    import torch
    import vllm
    print("✅ All key packages (transformers, torch, vllm) were imported successfully.")
    print(f"   - transformers: {transformers.__version__}")
    print(f"   - torch:        {torch.__version__}")
    print(f"   - vllm:         {vllm.__version__}")
except ImportError as e:
    print(f"❌ Failed to import a required package: {e}")
    print("   This likely means the installation cell failed or you are in the wrong kernel.")
    print("   Please re-run the setup cell, restart the kernel, and switch to the correct kernel.")

## 1b. Check GPU Status

Before running the rest of the notebook, it's crucial to select an available GPU. Run the cell below to see the current status of all GPUs on the server.

Look for a GPU with:
- **Low `Volatile GPU-Util`**: This indicates the GPU is not busy (look for `0%`).
- **High free memory**: The `Memory-Usage` should show a small number on the left (e.g., `10MiB`) and a large number on the right (e.g., `24564MiB`).

Once you find a free GPU, note its `ID` (the number on the far left) and set the `GPU_ID` variable in the configuration cell below accordingly.

In [ ]:
!nvidia-smi

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [ ]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "1"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 4096                 # Reduced from 32768 to prevent errors with max_model_len

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [ ]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [ ]:
# This cell is commented out to use the standard Transformers library instead of vLLM.
# vLLM build is failing due to incompatibility with the server's new CUDA 13 drivers.
print("Skipping vLLM model load.")

## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# The MODEL_ID is defined in the configuration cell above.

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16, # Use bfloat16 for modern GPUs
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Model loaded with Transformers and bitsandbytes.")

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [ ]:
# This cell is commented out as we are using the Transformers library for generation.
print("Skipping vLLM generation.")

### Generate with Transformers (for Datahub)

In [ ]:
# This cell generates responses using the Hugging Face Transformers library.
# It processes questions one by one to avoid memory issues from batch padding.
# This is slower than vLLM but more robust and compatible.

import torch
from tqdm import tqdm

responses = []
print(f"Generating responses for {len(data)} questions with Transformers...")

for item in tqdm(data, desc="Generating"):
    # 1. Build the prompt for the current item
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    
    # 2. Tokenize the single prompt
    inputs = tokenizer(prompt_text, return_tensors="pt").to(llm.device)
    
    # 3. Generate a response
    with torch.no_grad():
        output_ids = llm.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS,
            temperature=0.6,
            top_p=0.95,
            top_k=20,
            repetition_penalty=1.0,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id # Suppress warning
        )
        
    # 4. Decode only the new tokens
    input_len = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0, input_len:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    responses.append(response)

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [ ]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

## 8. Summary

Print accuracy broken down by question type.

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!